# 🏦 Camillion — One-Click Training (+ JARVIS)

Train **one bot that trades all four symbols from one account**, then read the results the way the
FTMO challenge is judged — **day by day** (did it make **+2.5%** and stay inside the **4% wall**).

**You only feed 1-minute candles.** This notebook builds everything else itself: it resamples your 1m
data up to **5m / 30m / 4h / 1d**, computes 44 indicators on each (220 total), and aligns them back to
the minute using only the *last closed* higher-timeframe bar (no future peeking).

**How to use:** run the cells top to bottom — click the ▶ on the left of each one.

_(If the clone cell asks for a username/password, the GitHub repo is private — make it public, or paste a
personal-access-token into the URL. Everything else is unchanged.)_

## Step 1 — Connect your Google Drive
A popup will ask permission: pick your account and click **Allow**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Get the bot's code
(Safe to re-run; it just updates to the latest.)

In [ ]:
!git clone -b feat/jarvis-bridge https://github.com/monty313/Camillion_Claude_RL_model.git /content/Camillion_Claude_RL_model 2>/dev/null || (cd /content/Camillion_Claude_RL_model && git pull --ff-only)
%cd /content/Camillion_Claude_RL_model

## Step 3 — Install the training engine
(~1 minute, only needed once per session.)

In [ ]:
!pip install -q stable-baselines3 torch

## Step 4 — (Optional, recommended) Health check
Wait for **✅ GO**. If it says 🚫 NO-GO it lists exactly what to fix first.

In [ ]:
!python tools/run_full_audit.py

## Step 5 — Put your 4 CSVs in Drive, then TRAIN

1. In Google Drive, make a folder named **`Camillion_data`** and drop your four 1-minute CSVs in it
   (filenames containing **EURUSD, GBPUSD, XAUUSD, US30**).
2. Run the cell below. It finds them, builds the features, trains the one portfolio bot, and prints the
   **day-by-day** +2.5% / 4%-wall report.

_Tweaks (optional): add `--steps 4000000` (longer/better), `--symbols EURUSD,XAUUSD`, or `--out models/my_bot`._

In [ ]:
!python run_training.py --data /content/drive/MyDrive/Camillion_data

## Step 6 — Open JARVIS (the read-only cockpit + advisor)

Run this to get a **clickable link** to JARVIS. It works even before you've trained (it shows an honest
synthetic demo). To point JARVIS at **your trained bot**, use the second (commented-out) line instead.

Ask JARVIS things like *“how did I do day by day?”*, *“is my bot safe to run?”*, *“which policy should I run?”*

In [ ]:
# === Open JARVIS (read-only cockpit + advisor) ===
# Renders JARVIS INLINE in this notebook (no external link, no DNS issues). Works even before
# training (synthetic demo). For YOUR trained bot, use the second Popen line (run AFTER Step 5).
!pip install -q -r requirements-jarvis.txt
import subprocess, time, urllib.request
subprocess.Popen(['python', 'go_live.py', '--port', '8000'])
# subprocess.Popen(['python','go_live.py','--port','8000',
#                   '--data','data_cache','--model','models/camillion_portfolio_ppo'])

# wait until the cockpit is actually listening
for _ in range(40):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=1); print('JARVIS is up \u2705'); break
    except Exception:
        time.sleep(1)
else:
    print('\u26a0\ufe0f JARVIS did not start \u2014 re-run this cell, or read the output above.')

from jarvis_bridge import COCKPIT_FILE
from google.colab import output
# 1) inline panel (most reliable):
output.serve_kernel_port_as_iframe(8000, path='/' + COCKPIT_FILE, height=900)
# 2) and a button to pop it out into its own tab:
output.serve_kernel_port_as_window(8000, path='/' + COCKPIT_FILE, anchor_text='Open JARVIS in a new tab')
